# 01 — Data Exploration & Dataset Acquisition (Non-Russia PastVu Supplement)

This copy is configured to supplement **Timeguessr AI** with **non-Russian PastVu historical photographs** by scanning middle/later shards of `nyuuzyou/pastvu` and filtering out `country == 'RU'`:
1. **Local Timeguessr daily challenges** (`data/images/Timeguessr/`)
2. **OSV-5M** ([`osv5m/osv5m`](https://huggingface.co/datasets/osv5m/osv5m)) — global street-view geolocation images
3. **PastVu (Non-RU)** ([`nyuuzyou/pastvu`](https://huggingface.co/datasets/nyuuzyou/pastvu)) — historical geotagged & year-labeled photographs outside Russia (`country != 'RU'`), saved to `data/images/pastvu_non_ru/`
4. **YFCC15M** ([`mehdidc/yfcc15m`](https://huggingface.co/datasets/mehdidc/yfcc15m)) — filtered strictly for **valid geolocation** and **valid capture date**

In [1]:
import torch

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch: 2.11.0+cu128 | CUDA available: True
Device: NVIDIA GeForce RTX 4070 SUPER
VRAM: 12.88 GB


## 0. Configuration & Directories

These datasets are very large at full scale (OSV-5M is ~250 GB, PastVu is ~2M images across 2,094 tar shards, and YFCC15M's metadata CSV alone is ~10 GB). Adjust the limits below to start with a manageable subset or set them to `None` to download everything.

> **Optional:** If you hit Hugging Face rate limits, set your `HF_TOKEN` in a `.env` file or call `huggingface_hub.login()`.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login, get_token, whoami

load_dotenv()

# --- Hugging Face Authentication (raises Hub/resolver rate limits) ---
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print(f"Authenticated with Hugging Face as: {whoami()['name']}")
elif get_token():
    print(f"Authenticated with cached HF token as: {whoami()['name']}")
else:
    print("Not authenticated with Hugging Face (using lower anonymous rate limits).")
    print("Set HF_TOKEN in .env or uncomment login() below to authenticate interactively:")
    # login()

# Base data directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
IMAGES_DIR = DATA_DIR / "images"

OSV5M_DIR = IMAGES_DIR / "osv5m"
PASTVU_DIR = IMAGES_DIR / "pastvu_non_ru"   # Separate folder so existing data/images/pastvu/ is preserved
YFCC_DIR = IMAGES_DIR / "yfcc15m"
TIMEGUESSR_DIR = IMAGES_DIR / "Timeguessr"

for d in (OSV5M_DIR, PASTVU_DIR, YFCC_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Global Year Bounds (Timeguessr range) ---
MIN_YEAR = 1900
MAX_YEAR = 2026

# PastVu (Non-RU): scan middle/later shards and exclude Russia to balance historical geography
PASTVU_MAX_SAMPLES = 15_000             # None = all matching non-RU images
PASTVU_EXCLUDE_COUNTRIES = {"RU"}       # ISO-2 country codes to exclude via offline reverse_geocoder
PASTVU_SHARD_START_INDEX = 800          # Skip earliest Moscow/Russia-heavy shards (out of 2,094 total)
PASTVU_SHUFFLE_SHARDS = True            # Deterministic shuffle over later shards for global diversity
PASTVU_MAX_PER_COUNTRY = None           # Optional per-country cap (e.g., 3_000) or None for uncapped

# --- Balanced Sampling Controls ---
# alpha = 0.0 -> strictly uniform | 0.5 -> square-root (sublinear) | 1.0 -> raw distribution
OSV5M_TARGET_SAMPLES = 50_000
OSV5M_COUNTRY_ALPHA = 0.6
OSV5M_MAX_PER_SEQUENCE = 2
OSV5M_MAX_ZIPS = 33          # More shards = better coverage of rare countries (set None for all 98)

# YFCC15M: number of valid (geotagged + valid date) rows to collect, and how many images to fetch
YFCC_MIN_GEO_ACCURACY = 11        # Flickr accuracy: 1=world, 3=country, 6=region, 11=city, 16=street
YFCC_DOWNLOAD_WORKERS = 16
YFCC_TARGET_IMAGES = 50_000
YFCC_COUNTRY_ALPHA = 0.6
YFCC_MAX_VALID_SCAN = 200_000     # Valid metadata rows to stream before sublinear selection
YFCC_OVERSAMPLE_FACTOR = 1.8      # Extra metadata per bucket to survive dead Flickr URLs
YFCC_GEOCODE_BATCH_SIZE = 4_000

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

Authenticated with cached HF token as: darksabre1500
Project root: d:\VSCodeProjects\CS441
Data dir:     d:\VSCodeProjects\CS441\data


## 2. Download `nyuuzyou/pastvu` (Non-Russia Historical Photos from Later Shards)

- Packaged on Hugging Face in WebDataset (`.tar`) format with paired images and JSON metadata (`geo: [lat, lon]`, `year`, `year2`, `title`, etc.).
- Skips the earliest Russia/Moscow-heavy shards (`PASTVU_SHARD_START_INDEX`) and deterministically shuffles the remaining middle/end shards (`2,094` total) for broad global coverage.
- Batch reverse-geocodes each shard's valid coordinates offline via `reverse_geocoder` (`mode=1` for Windows safety) and **filters out `country in PASTVU_EXCLUDE_COUNTRIES` (`{'RU'}`)** *before* writing JPEGs to disk.
- Saves images into `data/images/pastvu_non_ru/images/` and tabular labels (including `country`) into `data/images/pastvu_non_ru/metadata.csv`.

In [5]:
import json
import math
import random
import tarfile
from collections import Counter
from io import BytesIO
import pandas as pd
import reverse_geocoder as rg
from PIL import Image
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

PASTVU_REPO = "nyuuzyou/pastvu"
IMAGE_EXTS = {"jpg", "jpeg", "png", "webp"}

def _parse_pastvu_meta(meta: dict, min_year: int, max_year: int, max_year_span: int) -> dict | None:
    """Validate PastVu JSON metadata without touching disk yet (so RU photos are skipped before JPEG encoding)."""
    year = meta.get("year")
    year2 = meta.get("year2")
    try:
        year = int(year)
    except (TypeError, ValueError):
        return None
    if not (min_year <= year <= max_year):
        return None
    if year2 is not None:
        try:
            year2 = int(year2)
            if abs(year2 - year) > max_year_span:
                return None
            year = round((year + year2) / 2)
        except (TypeError, ValueError):
            year2 = None

    geo = meta.get("geo")
    if not isinstance(geo, (list, tuple)) or len(geo) != 2:
        return None
    try:
        lat, lon = float(geo[0]), float(geo[1])
    except (TypeError, ValueError):
        return None
    if not (math.isfinite(lat) and math.isfinite(lon)):
        return None
    if not (-90.0 <= lat <= 90.0 and -180.0 <= lon <= 180.0) or (lat == 0.0 and lon == 0.0):
        return None

    cid = str(meta.get("cid") or "unknown")
    return {
        "id": cid,
        "year": year,
        "year_start": meta.get("year"),
        "year_end": year2,
        "latitude": lat,
        "longitude": lon,
        "title": meta.get("title", ""),
        "source": "pastvu_non_ru",
    }

def download_pastvu(
    output_dir: Path = PASTVU_DIR,
    max_samples: int | None = PASTVU_MAX_SAMPLES,
    min_year: int = MIN_YEAR,
    max_year: int = MAX_YEAR,
    max_year_span: int = 5,
    exclude_countries: set[str] | None = PASTVU_EXCLUDE_COUNTRIES,
    shard_start_index: int = PASTVU_SHARD_START_INDEX,
    shuffle_shards: bool = PASTVU_SHUFFLE_SHARDS,
    max_per_country: int | None = PASTVU_MAX_PER_COUNTRY,
    seed: int = 42,
    keep_tars: bool = False,
) -> pd.DataFrame:
    img_dir = output_dir / "images"
    shard_dir = output_dir / "shards"
    img_dir.mkdir(parents=True, exist_ok=True)
    shard_dir.mkdir(parents=True, exist_ok=True)
    meta_path = output_dir / "metadata.csv"
    state_path = output_dir / "completed_shards.json"
    excluded = {c.upper() for c in (exclude_countries or set())}

    # Resume from existing metadata.csv and completed_shards.json if present
    records: list[dict] = []
    seen_ids: set[str] = set()
    country_counts: Counter = Counter()
    completed_shards: set[str] = set()
    if meta_path.exists():
        prev_df = pd.read_csv(meta_path, dtype={"id": str})
        records = prev_df.to_dict(orient="records")
        seen_ids = set(prev_df["id"].astype(str))
        if "country" in prev_df.columns:
            country_counts.update(prev_df["country"].dropna().astype(str).str.upper())
        print(f"Resuming PastVu (non-RU) from {len(records):,} existing records in {meta_path}")
    if state_path.exists():
        completed_shards = set(json.loads(state_path.read_text(encoding="utf-8")))
        print(f"Skipping {len(completed_shards)} already-completed tar shards.")

    api = HfApi()
    all_tar_files = sorted(
        f for f in api.list_repo_files(repo_id=PASTVU_REPO, repo_type="dataset")
        if f.endswith(".tar")
    )
    tar_files = all_tar_files[shard_start_index:]
    if shuffle_shards:
        rng = random.Random(seed)
        rng.shuffle(tar_files)
    print(
        f"Found {len(all_tar_files):,} total PastVu tar shards; "
        f"scanning {len(tar_files):,} shards from index {shard_start_index} "
        f"(shuffled={shuffle_shards}, excluding={sorted(excluded)})."
    )

    pbar = tqdm(total=max_samples, initial=len(records), desc="PastVu non-RU samples")
    skipped_excluded = 0

    try:
        for tar_rel in tar_files:
            if max_samples is not None and len(records) >= max_samples:
                break
            if tar_rel in completed_shards:
                continue

            try:
                tar_path = Path(
                    hf_hub_download(
                        repo_id=PASTVU_REPO,
                        repo_type="dataset",
                        filename=tar_rel,
                        local_dir=str(shard_dir),
                    )
                )
            except Exception as exc:
                print(f"Warning: failed to download shard {tar_rel} ({exc}); skipping.")
                continue

            # 1. Read shard pairs and validate metadata in memory
            pending: dict[str, dict] = {}
            candidates: list[tuple[dict, bytes]] = []
            try:
                with tarfile.open(tar_path, "r:*") as tf:
                    for member in tf:
                        if not member.isfile() or "." not in member.name:
                            continue
                        stem, ext = member.name.rsplit(".", 1)
                        ext = ext.lower()
                        if ext != "json" and ext not in IMAGE_EXTS:
                            continue

                        fobj = tf.extractfile(member)
                        if fobj is None:
                            continue
                        raw = fobj.read()
                        entry = pending.setdefault(stem, {})
                        if ext == "json":
                            try:
                                entry["meta"] = json.loads(raw)
                            except Exception:
                                pass
                        else:
                            entry["img_bytes"] = raw

                        if "meta" in entry and "img_bytes" in entry:
                            pending.pop(stem)
                            rec = _parse_pastvu_meta(entry["meta"], min_year, max_year, max_year_span)
                            if rec is not None and rec["id"] not in seen_ids:
                                candidates.append((rec, entry["img_bytes"]))
            except (tarfile.TarError, EOFError, OSError) as exc:
                print(f"Warning: corrupted/truncated shard {tar_rel} ({exc}); deleting and skipping.")
                if tar_path.exists():
                    tar_path.unlink(missing_ok=True)
                completed_shards.add(tar_rel)
                state_path.write_text(json.dumps(sorted(completed_shards)), encoding="utf-8")
                continue

            # 2. Batch reverse-geocode all valid candidates in this shard, then filter out RU
            if candidates:
                coords = [(r["latitude"], r["longitude"]) for r, _ in candidates]
                geo_results = rg.search(coords, mode=1)

                for (rec, img_bytes), geo in zip(candidates, geo_results):
                    if max_samples is not None and len(records) >= max_samples:
                        break
                    cc = str(geo.get("cc", "")).upper()
                    if len(cc) != 2 or cc in excluded:
                        skipped_excluded += 1
                        continue
                    if max_per_country is not None and country_counts[cc] >= max_per_country:
                        continue
                    if rec["id"] in seen_ids:
                        continue

                    img_path = img_dir / f"{rec['id']}.jpg"
                    if not img_path.exists():
                        try:
                            with Image.open(BytesIO(img_bytes)) as im:
                                im.convert("RGB").save(img_path, format="JPEG", quality=92)
                        except Exception:
                            continue

                    rec["country"] = cc
                    rec["image_path"] = str(img_path.relative_to(PROJECT_ROOT))
                    seen_ids.add(rec["id"])
                    country_counts[cc] += 1
                    records.append(rec)
                    pbar.update(1)

            pbar.set_postfix(countries=len(country_counts), skipped_ru=skipped_excluded)
            completed_shards.add(tar_rel)
            if not keep_tars and tar_path.exists():
                tar_path.unlink()

            # Checkpoint after every shard
            if records:
                pd.DataFrame(records).to_csv(meta_path, index=False)
            state_path.write_text(json.dumps(sorted(completed_shards)), encoding="utf-8")

    except KeyboardInterrupt:
        print("\nInterrupted! Saving collected PastVu (non-RU) metadata so far...")
    finally:
        pbar.close()
        df = pd.DataFrame(records)
        if len(df) > 0:
            df.to_csv(meta_path, index=False)
            state_path.write_text(json.dumps(sorted(completed_shards)), encoding="utf-8")
            top5 = dict(country_counts.most_common(5))
            print(
                f"Saved {len(df):,} non-RU PastVu samples across {df['country'].nunique()} countries "
                f"(skipped {skipped_excluded:,} excluded) -> {meta_path}\nTop 5 countries: {top5}"
            )
        else:
            print("No non-RU PastVu samples collected yet.")

    return df

pastvu_df = download_pastvu()
display(pastvu_df.head())

Resuming PastVu (non-RU) from 6,782 existing records in d:\VSCodeProjects\CS441\data\pastvu_non_ru\metadata.csv
Skipping 20 already-completed tar shards.
Found 2,094 total PastVu tar shards; scanning 1,294 shards from index 800 (shuffled=True, excluding=['RU']).


PastVu non-RU samples:  45%|####5     | 6782/15000 [00:00<?, ?it/s]

pastvu_001655.tar: reconstructing file:   0%|          |  0.00B /  519MB            

pastvu_001655.tar: downloading bytes:           |  0.00B            

pastvu_001555.tar: reconstructing file:   0%|          |  0.00B / 1.13GB            

pastvu_001555.tar: downloading bytes:           |  0.00B            

... (37 outputs left out)

pastvu_001505.tar: reconstructing file:   0%|          |  0.00B / 3.57GB            

pastvu_001505.tar: downloading bytes:           |  0.00B            

Saved 15,000 non-RU PastVu samples across 63 countries (skipped 8,928 excluded) -> d:\VSCodeProjects\CS441\data\pastvu_non_ru\metadata.csv
Top 5 countries: {'UA': 4113, 'DE': 1933, 'US': 1597, 'FR': 1296, 'UZ': 1264}


,id,year,year_start,year_end,latitude,longitude,title,source,country,image_path
0,1095122,1991,1991,1991,48.673750,26.561256,Каменец-Подольский замок. Вид с западной стороны,pastvu_non_ru,UA,data\pastvu_non_ru\images\1095122.jpg
1,1095157,1991,1991,1991,48.675582,26.572692,Ратуша,pastvu_non_ru,UA,data\pastvu_non_ru\images\1095157.jpg
2,1096621,1991,1991,1991,48.679932,26.573308,Пивоваренный завод. Вид сквозь проём в Турецко...,pastvu_non_ru,UA,data\pastvu_non_ru\images\1096621.jpg
3,1096622,1991,1991,1991,48.674636,26.561218,Вид на Польские Фольварки,pastvu_non_ru,UA,data\pastvu_non_ru\images\1096622.jpg
4,1096623,1991,1991,1991,48.673584,26.563863,Вид на Польские Фольварки от Замковой улицы,pastvu_non_ru,UA,data\pastvu_non_ru\images\1096623.jpg
